# Spinal Disorder Classification — Optimization Research

## Stage 1: Dataset inspection and preprocessing audit

**Goal:** audit the original X-rays before making a train/validation/test split or training a new neural network. Preserve the original research notebook separately.

Expected folder structure in Google Drive:

```text
MyDrive/224/
  Normal/    # label 0
  Scol/      # label 1 (scoliosis)
  Spond/     # label 2 (spondylolisthesis)
```

Earlier Google Drive metadata showed **71 Normal, 188 Scol, and 79 Spond entries (338 total)**. The old notebook loaded 336 because it selected only `.jpg`/`.JPG` paths; this notebook checks supported image files by **content**, not just extension.

**Privacy and research limitations:** Only use X-rays you are authorized to process, and keep all potentially identifying images/filenames in private storage. This is an educational research project, not a medical diagnostic tool. **Do not upload the image dataset or the private image manifest to your public GitHub repository.**

**Run the notebook top to bottom.** No data is modified or deleted, and no model training occurs during Stage 1.

### 1. Connect Google Drive

The first cell requires your permission to mount Drive. The subsequent cells only read images from `224/` and save reports to a separate results folder.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

### 2. Imports, class labels, and paths

We keep the numerical class labels consistent with your original project. Adjust `DATA_ROOT` only if your `224` folder moved.

In [ ]:
from pathlib import Path
from collections import Counter
import hashlib
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATA_ROOT = Path('/content/drive/MyDrive/224')
AUDIT_DIR = Path('/content/drive/MyDrive/Spinal_Classification_Optimization/audit')
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

CLASS_TO_LABEL = {'Normal': 0, 'Scol': 1, 'Spond': 2}
CLASS_DISPLAY = {'Normal': 'Normal', 'Scol': 'Scoliosis', 'Spond': 'Spondylolisthesis'}
SUPPORTED_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

assert DATA_ROOT.is_dir(), f'Dataset folder not found: {DATA_ROOT}'
for class_name in CLASS_TO_LABEL:
    assert (DATA_ROOT / class_name).is_dir(), f'Missing folder: {DATA_ROOT / class_name}'

print('Dataset:', DATA_ROOT)
print('Private audit output:', AUDIT_DIR)

### 3. Audit each image without changing it

For every file, we record dimensions, encoding, aspect ratio, file checksum, decoded-pixel checksum, and a small perceptual difference-hash. `PIL` detects the **actual image format**, even when the filename extension is misleading.

The perceptual hash is only a **screening aid**: a close match can be two different medical images. We will never automatically delete images based on it.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def decoded_pixel_hash(rgb_image):
    digest = hashlib.sha256()
    digest.update(str(rgb_image.size).encode('utf-8'))
    digest.update(rgb_image.tobytes())
    return digest.hexdigest()


def difference_hash(rgb_image):
    # 64-bit grayscale difference hash for possible near-duplicates.
    small = ImageOps.grayscale(rgb_image).resize((9, 8), Image.Resampling.LANCZOS)
    pixels = np.asarray(small, dtype=np.uint8)
    comparisons = pixels[:, 1:] > pixels[:, :-1]
    value = 0
    for bit in comparisons.flat:
        value = (value << 1) | int(bit)
    return f'{value:016x}'


def audit_image(path, class_name):
    row = {
        'relative_path': str(path.relative_to(DATA_ROOT)),
        'filename': path.name,
        'class_name': class_name,
        'label': CLASS_TO_LABEL[class_name],
        'extension': path.suffix.lower(),
        'file_size_bytes': path.stat().st_size,
        'status': 'ok',
        'error': '',
    }

    try:
        with Image.open(path) as check:
            detected_format = check.format
            check.verify()
        with Image.open(path) as opened:
            rgb = ImageOps.exif_transpose(opened).convert('RGB')

        pixels = np.asarray(rgb)
        height, width = pixels.shape[:2]
        is_gray = bool(
            np.array_equal(pixels[:, :, 0], pixels[:, :, 1])
            and np.array_equal(pixels[:, :, 1], pixels[:, :, 2])
        )
        row.update({
            'detected_format': detected_format,
            'width': width,
            'height': height,
            'aspect_ratio': round(width / height, 4),
            'grayscale_content': is_gray,
            'file_sha256': sha256_file(path),
            'pixel_sha256': decoded_pixel_hash(rgb),
            'dhash64': difference_hash(rgb),
        })
    except Exception as exc:
        row.update({'status': 'error', 'error': f'{type(exc).__name__}: {exc}'})
    return row


rows = []
unexpected_files = []
for class_name in CLASS_TO_LABEL:
    class_dir = DATA_ROOT / class_name
    for path in sorted(class_dir.rglob('*')):
        if not path.is_file():
            continue
        if path.suffix.lower() not in SUPPORTED_EXTENSIONS:
            unexpected_files.append(str(path.relative_to(DATA_ROOT)))
            continue
        rows.append(audit_image(path, class_name))

manifest = pd.DataFrame(rows)
assert not manifest.empty, 'No supported images found. Check your Drive folder.'
valid = manifest[manifest['status'] == 'ok'].copy()
errors = manifest[manifest['status'] != 'ok'].copy()

print(f'Found {len(manifest)} supported image files')
print(f'Readable: {len(valid)} | Failed verification: {len(errors)}')
print(f'Unexpected non-image extensions: {len(unexpected_files)}')
if unexpected_files:
    print('Examples:', unexpected_files[:10])
if not errors.empty:
    display(errors[['relative_path', 'error']])

### 4. Check class counts, resolutions, and extension mismatches

The `.png` files found in Drive may be genuine PNGs or merely JPEGs named `.png`. Both are loadable through PIL, but we want to document what is actually present.

In [ ]:
counts = (
    valid.groupby(['class_name', 'label'])
    .size()
    .reindex(pd.MultiIndex.from_tuples([(name, label) for name, label in CLASS_TO_LABEL.items()], names=['class_name', 'label']), fill_value=0)
    .reset_index(name='images')
)
counts['display_name'] = counts['class_name'].map(CLASS_DISPLAY)
display(counts[['display_name', 'label', 'images']])
print('Total verified images:', len(valid))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(counts['display_name'], counts['images'])
axes[0].set_title('Dataset class balance')
axes[0].set_ylabel('Images')
axes[0].tick_params(axis='x', rotation=10)
axes[1].scatter(valid['width'], valid['height'], alpha=0.55)
axes[1].set_title('Original image dimensions')
axes[1].set_xlabel('Width (pixels)')
axes[1].set_ylabel('Height (pixels)')
plt.tight_layout()
plt.show()

print('Resolution summary:')
display(valid[['width', 'height', 'aspect_ratio', 'file_size_bytes']].describe().round(2))
print('File extension / actual encoding:')
display(pd.crosstab(valid['extension'], valid['detected_format']))
print('Images whose three color channels are identical:', int(valid['grayscale_content'].sum()), '/', len(valid))

EXPECTED_FORMAT = {
    '.jpg': 'JPEG', '.jpeg': 'JPEG', '.png': 'PNG',
    '.bmp': 'BMP', '.tif': 'TIFF', '.tiff': 'TIFF', '.webp': 'WEBP',
}
format_mismatches = valid[
    valid.apply(lambda r: EXPECTED_FORMAT.get(r['extension']) != r['detected_format'], axis=1)
].copy()
print('Extension / detected-format mismatches:', len(format_mismatches))
if not format_mismatches.empty:
    display(format_mismatches[['relative_path', 'extension', 'detected_format']])

### 5. Visual inspection: random samples from each class

Inspect for non-spine images, cropping problems, text overlays, inconsistent orientation, or images that look suspiciously alike. A random selection is just a first look, not a comprehensive medical-quality audit.

In [ ]:
rng = np.random.default_rng(SEED)
fig, axes = plt.subplots(3, 4, figsize=(13, 12))
for row_i, class_name in enumerate(CLASS_TO_LABEL):
    paths = valid.loc[valid['class_name'] == class_name, 'relative_path'].tolist()
    chosen = rng.choice(paths, size=min(4, len(paths)), replace=False) if paths else []
    for col_i, ax in enumerate(axes[row_i]):
        ax.axis('off')
        if col_i >= len(chosen):
            continue
        rel = str(chosen[col_i])
        with Image.open(DATA_ROOT / rel) as im:
            picture = ImageOps.exif_transpose(im).convert('RGB')
            picture.thumbnail((350, 350))
            ax.imshow(picture)
        ax.set_title(f'{CLASS_DISPLAY[class_name]}\n{Path(rel).name[:30]}', fontsize=9)
plt.suptitle('Sample images by class', fontsize=14)
plt.tight_layout()
plt.show()

### 6. Screen for exact and near-duplicate images

- **File checksum match:** byte-for-byte duplicate.
- **Pixel checksum match:** identical decoded RGB image, potentially different file encodings.
- **Small perceptual-hash distance:** *possible* near-duplicate, requiring human review.

We intentionally **do not remove anything automatically**. Check possible overlaps between classes especially carefully, and check whether multiple images belong to the same patient before splitting.

In [ ]:
exact_file_duplicates = valid[valid.duplicated('file_sha256', keep=False)].sort_values('file_sha256')
exact_pixel_duplicates = valid[valid.duplicated('pixel_sha256', keep=False)].sort_values('pixel_sha256')

candidate_pairs = []
items = valid[['relative_path', 'class_name', 'pixel_sha256', 'dhash64']].to_dict('records')
NEAR_DUPLICATE_DISTANCE = 4
for i in range(len(items)):
    a = items[i]
    for j in range(i + 1, len(items)):
        b = items[j]
        if a['pixel_sha256'] == b['pixel_sha256']:
            continue  # Already covered by exact decoded-pixel matches.
        distance = (int(a['dhash64'], 16) ^ int(b['dhash64'], 16)).bit_count()
        if distance <= NEAR_DUPLICATE_DISTANCE:
            candidate_pairs.append({
                'image_a': a['relative_path'], 'class_a': a['class_name'],
                'image_b': b['relative_path'], 'class_b': b['class_name'],
                'dhash_distance': distance,
                'cross_class': a['class_name'] != b['class_name'],
            })

near_duplicates = pd.DataFrame(candidate_pairs, columns=[
    'image_a', 'class_a', 'image_b', 'class_b', 'dhash_distance', 'cross_class'
])
if not near_duplicates.empty:
    near_duplicates = near_duplicates.sort_values(['cross_class', 'dhash_distance'], ascending=[False, True])

print('Images in byte-exact duplicate groups:', len(exact_file_duplicates))
print('Images in pixel-exact duplicate groups:', len(exact_pixel_duplicates))
print('Possible near-duplicate pairs:', len(near_duplicates))
if not exact_pixel_duplicates.empty:
    display(exact_pixel_duplicates[['relative_path', 'class_name', 'pixel_sha256']].head(20))
if not near_duplicates.empty:
    display(near_duplicates.head(20))

### 7. Optional: review a candidate near-duplicate pair

Change `pair_index` to inspect another pair. **Similarity does not establish that two X-rays belong to the same person.**

In [ ]:
if near_duplicates.empty:
    print('No candidate near-duplicate pairs at the current dHash threshold.')
else:
    pair_index = 0  # Change this index to inspect other rows.
    pair = near_duplicates.iloc[pair_index]
    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    for ax, col, title in zip(axes, ['image_a', 'image_b'], ['Candidate A', 'Candidate B']):
        path = DATA_ROOT / pair[col]
        with Image.open(path) as im:
            ax.imshow(ImageOps.exif_transpose(im).convert('RGB'))
        ax.set_title(f'{title}\n{pair[col]}', fontsize=9)
        ax.axis('off')
    plt.suptitle(f'Perceptual hash distance: {pair["dhash_distance"]}')
    plt.tight_layout()
    plt.show()

### 8. Preview preprocessing candidates at 128 × 128

We will later **compare** RGB, grayscale, and contrast-limited adaptive histogram equalization (CLAHE). This cell previews the options but **does not write processed files or train a model**.

- `ImageOps.pad` preserves the original aspect ratio, padding around the X-ray rather than stretching anatomy.
- Grayscale converts the image to one channel.
- CLAHE enhances local contrast but can also amplify imaging artifacts.
- Every eventual training and website-inference pipeline must apply **exactly the same preprocessing**.

In [ ]:
import cv2  # OpenCV is available in standard Colab runtimes.


def make_preprocessing_preview(path, size=(128, 128)):
    with Image.open(path) as opened:
        original = ImageOps.exif_transpose(opened).convert('RGB')
        rgb = ImageOps.pad(original, size, method=Image.Resampling.LANCZOS, color=(0, 0, 0))
    rgb_array = np.asarray(rgb, dtype=np.uint8)
    gray = cv2.cvtColor(rgb_array, cv2.COLOR_RGB2GRAY)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(gray)
    return rgb_array, gray, clahe

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for i, class_name in enumerate(CLASS_TO_LABEL):
    sample = valid.loc[valid['class_name'] == class_name, 'relative_path'].iloc[0]
    rgb, gray, clahe = make_preprocessing_preview(DATA_ROOT / sample)
    for j, (image, title) in enumerate(zip((rgb, gray, clahe), ('RGB', 'Grayscale', 'CLAHE'))):
        axes[i, j].imshow(image, cmap='gray' if j else None, vmin=0 if j else None, vmax=255 if j else None)
        axes[i, j].set_title(f'{CLASS_DISPLAY[class_name]} — {title}')
        axes[i, j].axis('off')
plt.suptitle('Preprocessing candidates (visual preview only)')
plt.tight_layout()
plt.show()

print('For neural networks, normalized arrays will be float32 in the [0, 1] range.')
print('RGB shape: (128, 128, 3); grayscale / CLAHE shape: (128, 128, 1) after channel expansion.')

### 9. Save the Stage 1 audit (privately)

The full manifest contains source filenames; it stays in your private Google Drive. Do **not** commit it to GitHub unless you have reviewed it for sensitive patient information. Only aggregate, non-identifying statistics should be published.

In [ ]:
manifest.to_csv(AUDIT_DIR / 'manifest_full_PRIVATE.csv', index=False)
errors.to_csv(AUDIT_DIR / 'unreadable_files_PRIVATE.csv', index=False)
format_mismatches.to_csv(AUDIT_DIR / 'format_mismatches_PRIVATE.csv', index=False)
exact_file_duplicates.to_csv(AUDIT_DIR / 'byte_duplicates_PRIVATE.csv', index=False)
exact_pixel_duplicates.to_csv(AUDIT_DIR / 'pixel_duplicates_PRIVATE.csv', index=False)
near_duplicates.to_csv(AUDIT_DIR / 'near_duplicate_candidates_PRIVATE.csv', index=False)

public_summary = {
    'stage': 'dataset_audit_only',
    'seed': SEED,
    'total_supported_files': int(len(manifest)),
    'readable_images': int(len(valid)),
    'unreadable_images': int(len(errors)),
    'images_by_class': {CLASS_DISPLAY[name]: int((valid['class_name'] == name).sum()) for name in CLASS_TO_LABEL},
    'detected_formats': {str(k): int(v) for k, v in valid['detected_format'].value_counts().items()},
    'potential_extension_format_mismatches': int(len(format_mismatches)),
    'files_in_byte_duplicate_groups': int(len(exact_file_duplicates)),
    'files_in_pixel_duplicate_groups': int(len(exact_pixel_duplicates)),
    'candidate_near_duplicate_pairs': int(len(near_duplicates)),
    'evaluation_status': 'not_split_or_trained',
}
with open(AUDIT_DIR / 'dataset_summary_aggregate.json', 'w', encoding='utf-8') as handle:
    json.dump(public_summary, handle, indent=2)

print('Audit saved to:', AUDIT_DIR)
print(json.dumps(public_summary, indent=2))

## Stage 1 checkpoint — review before training

Before moving to Stage 2, we will:

1. Review unreadable or mislabeled files and duplicate candidates, including any possible cross-class conflicts.
2. Identify whether filenames or other metadata provide **patient IDs**; if so, group images from the same patient into the same split.
3. If patient identity cannot be established, **document that limitation** and avoid describing the resulting test set as patient-independent.
4. Freeze the cleaned dataset manifest, class mapping, random seed, and split strategy. Create the test split only once and never use it for preprocessing/model selection.
5. Build preprocessing and training pipelines using only training-derived statistics, compare validation results, and save model checkpoints and configurations.

**Do not optimize model accuracy on the final test set.** Once the audit is reviewed, Stage 2 will implement reproducible, leakage-aware splitting; Stage 3 will train our CNN experiments and compare grayscale/CLAHE and regularization strategies.

## Stage 2: Freeze leakage-aware dataset partitions

**Research protocol**

- Use the 338 verified images from Stage 1, with no automatic deletion.
- `N1`, `N2`, etc. are **not** accepted as patient identifiers. Patient independence cannot be established.
- Exact matches and Stage 1 near-duplicate candidates are assigned to the **same** partition. This is a conservative safeguard, not proof of patient identity.
- Any candidate pair spanning **different diagnostic classes** stops the process for manual review.
- Use class-stratified, group-aware 70/15/15 splitting (approximately, because groups must stay together).
- Save one private, immutable `v1` split in Drive. Re-running checks and reuses it; changes to source images or candidate pairs raise an error.
- Do not inspect test images or use test results to choose preprocessing, architectures, or hyperparameters.
- **GitHub:** Only upload source notebooks with cleared outputs and aggregate statistics. Never publish identifiable image filenames, the raw dataset, or private split manifests.

Only run these Stage 2 cells **after** completing the Stage 1 audit. They can also be run in a fresh Colab session if the audit CSVs were saved successfully.

### 10. Load and verify the Stage 1 audit

This checks every original file's checksum against the saved audit. If an image changed, the split will **not** proceed.

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from IPython.display import display

SEED = 42
TEST_FRACTION = 0.15
VALIDATION_FRACTION = 0.15

DATA_ROOT = Path("/content/drive/MyDrive/224")
AUDIT_DIR = Path("/content/drive/MyDrive/Spinal_Classification_Optimization/audit")
SPLIT_DIR = Path("/content/drive/MyDrive/Spinal_Classification_Optimization/splits/v1")

CLASS_TO_LABEL = {"Normal": 0, "Scol": 1, "Spond": 2}

audit_file = AUDIT_DIR / "manifest_full_PRIVATE.csv"
pairs_file = AUDIT_DIR / "near_duplicate_candidates_PRIVATE.csv"

if not audit_file.is_file() or not pairs_file.is_file():
    raise FileNotFoundError("Missing Stage 1 audit CSVs. Run the audit and save its reports first.")

audit = pd.read_csv(audit_file, keep_default_na=False)
pairs = pd.read_csv(pairs_file, keep_default_na=False)

required = {"relative_path", "class_name", "label", "status", "file_sha256", "pixel_sha256"}
assert required.issubset(audit.columns), f"Audit missing: {sorted(required - set(audit.columns))}"
assert (audit["status"] == "ok").all(), "Stop: audit has unreadable images."
assert audit["relative_path"].is_unique, "Stop: repeated file paths in audit."
assert len(audit) == 338, f"Expected 338 images; audit has {len(audit)}."

audit = audit.sort_values("relative_path").reset_index(drop=True)
assert set(audit["class_name"]) == set(CLASS_TO_LABEL)
assert all(
    int(row.label) == CLASS_TO_LABEL[row.class_name]
    for row in audit.itertuples(index=False)
), "Unexpected class-to-label mapping."

def file_checksum(path):
    h = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def content_digest(records):
    payload = json.dumps(records, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

# Fail if any file was added, removed, moved, or changed after Stage 1.
supported = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}
actual_paths = {
    str(path.relative_to(DATA_ROOT))
    for folder in CLASS_TO_LABEL
    for path in (DATA_ROOT / folder).rglob("*")
    if path.is_file() and path.suffix.lower() in supported
}
assert actual_paths == set(audit["relative_path"]), (
    "Dataset file list changed since Stage 1; review and re-audit before continuing."
)

for row in audit.itertuples(index=False):
    original = DATA_ROOT / row.relative_path
    assert file_checksum(original) == row.file_sha256, (
        "An image changed since Stage 1. Re-audit before splitting."
    )

dataset_fingerprint = content_digest(
    audit[["relative_path", "class_name", "label", "file_sha256"]]
    .to_dict(orient="records")
)
print("Verified unchanged dataset:", len(audit), "images")
print("SHA-256 manifest fingerprint:", dataset_fingerprint)


### 11. Construct conservative near-duplicate groups

Both suspected pairs stay together; repeated `N` numbers alone are **not** grouped. No images are deleted.

In [ ]:
# Group only actual duplicate evidence, not matching N-prefixes.
parent = {path: path for path in audit["relative_path"]}

def root(item):
    while parent[item] != item:
        parent[item] = parent[parent[item]]
        item = parent[item]
    return item

def union(a, b):
    ra, rb = root(a), root(b)
    if ra != rb:
        parent[max(ra, rb)] = min(ra, rb)

# Exact matches are automatically grouped, if any are present.
for field in ("file_sha256", "pixel_sha256"):
    for _, entries in audit.groupby(field):
        members = entries["relative_path"].tolist()
        for other in members[1:]:
            union(members[0], other)

pair_columns = {"image_a", "image_b", "class_a", "class_b"}
assert pair_columns.issubset(pairs.columns), (
    "Near-duplicate report is missing expected columns."
)
labels_by_path = audit.set_index("relative_path")["class_name"].to_dict()
pair_records = []

for pair in pairs.itertuples(index=False):
    a, b = str(pair.image_a), str(pair.image_b)
    assert a in parent and b in parent, "Candidate pair refers to an unknown image."
    assert labels_by_path[a] == pair.class_a and labels_by_path[b] == pair.class_b
    if labels_by_path[a] != labels_by_path[b]:
        raise ValueError(
            "A cross-class near-duplicate needs manual review before splitting."
        )
    union(a, b)
    pair_records.append({
        "image_a": min(a, b), "image_b": max(a, b),
        "class": labels_by_path[a],
    })

pair_records = sorted(pair_records, key=lambda item: (item["image_a"], item["image_b"]))
pairs_fingerprint = content_digest(pair_records)

audit["group_id"] = audit["relative_path"].map(root)

group_table = (
    audit.groupby("group_id")
    .agg(class_name=("class_name", "first"),
         label=("label", "first"),
         image_count=("relative_path", "size"),
         n_classes=("class_name", "nunique"))
    .reset_index()
)
assert (group_table["n_classes"] == 1).all(), (
    "A grouped image set contains conflicting labels; review manually."
)

print("Candidate near-duplicate pairs:", len(pair_records))
print("Groups after linking possible duplicates:", len(group_table))
print("Grouped images by largest group size:")
print(group_table["image_count"].value_counts().sort_index().to_string())


### 12. Stratified group splitting

Use 70% training, 15% validation, and 15% test **as targets**; the exact image counts may differ slightly to keep linked candidates together.

In [ ]:
# Split groups, not images. Group labels are unambiguous after the checks above.
# Fractions are approximate at the image level because linked images cannot be separated.
development_groups, test_groups = train_test_split(
    group_table,
    test_size=TEST_FRACTION,
    stratify=group_table["label"],
    random_state=SEED,
)

train_groups, validation_groups = train_test_split(
    development_groups,
    test_size=VALIDATION_FRACTION / (1.0 - TEST_FRACTION),
    stratify=development_groups["label"],
    random_state=SEED + 1,
)

split_lookup = {
    **{g: "train" for g in train_groups["group_id"]},
    **{g: "validation" for g in validation_groups["group_id"]},
    **{g: "test" for g in test_groups["group_id"]},
}
assert len(split_lookup) == len(group_table)

split_manifest = audit.copy()
split_manifest["split"] = split_manifest["group_id"].map(split_lookup)
assert split_manifest["split"].notna().all()

# Guard against any near-duplicate crossing a partition boundary.
for entry in pair_records:
    a_split = split_manifest.set_index("relative_path").at[entry["image_a"], "split"]
    b_split = split_manifest.set_index("relative_path").at[entry["image_b"], "split"]
    assert a_split == b_split, "Near-duplicate leakage between partitions!"

assert split_manifest.groupby("group_id")["split"].nunique().max() == 1
summary = (
    pd.crosstab(split_manifest["class_name"], split_manifest["split"])
    .reindex(index=list(CLASS_TO_LABEL), columns=["train", "validation", "test"], fill_value=0)
)
assert (summary > 0).all().all(), "At least one class is absent from a split."

display(summary.assign(total=summary.sum(axis=1)))
print("Total:", summary.sum().sum(), "images")
print("Fractions:", (split_manifest["split"].value_counts(normalize=True) * 100).round(1).to_dict())


### 13. Freeze split v1 to private Google Drive

**Run exactly once.** Later executions verify that the existing split is identical instead of overwriting it. Keep all `*_PRIVATE.*` artifacts off public GitHub.

In [ ]:
config = {
    "stage": "split_frozen_v1",
    "seed": SEED,
    "target_fractions": {
        "train": 1 - TEST_FRACTION - VALIDATION_FRACTION,
        "validation": VALIDATION_FRACTION,
        "test": TEST_FRACTION,
    },
    "class_to_label": CLASS_TO_LABEL,
    "dataset_fingerprint": dataset_fingerprint,
    "candidate_pair_fingerprint": pairs_fingerprint,
    "split_method": "stratified groups; 2-stage sklearn train_test_split",
    "grouping_rule": "exact matches and saved Stage 1 near-duplicate candidates only",
    "patient_independent": False,
    "limitation": "Patient identifiers not verified; source and acquisition leakage may remain.",
    "n_images": int(len(split_manifest)),
    "n_groups": int(len(group_table)),
}
summary_public = {
    "version": "v1",
    "seed": SEED,
    "patient_independent": False,
    "image_counts": {
        name: {split: int(summary.at[name, split]) for split in summary.columns}
        for name in summary.index
    },
}

private_csv = SPLIT_DIR / "split_manifest_PRIVATE.csv"
private_json = SPLIT_DIR / "split_config_PRIVATE.json"
public_json = SPLIT_DIR / "split_summary_AGGREGATE.json"

if SPLIT_DIR.exists():
    # An existing freeze is immutable. Refuse to create a new split silently.
    if not private_csv.exists() or not private_json.exists():
        raise RuntimeError(
            "Split folder exists but its freeze is incomplete. Inspect it manually; "
            "do not delete or overwrite it automatically."
        )
    frozen_config = json.loads(private_json.read_text(encoding="utf-8"))
    frozen_manifest = pd.read_csv(private_csv, keep_default_na=False)
    if frozen_config != config:
        raise RuntimeError("Existing v1 config differs. Do not overwrite the original split.")
    required_cols = ["relative_path", "class_name", "label", "file_sha256", "group_id", "split"]
    frozen_rows = frozen_manifest[required_cols].sort_values("relative_path").reset_index(drop=True)
    current_rows = split_manifest[required_cols].sort_values("relative_path").reset_index(drop=True)
    pd.testing.assert_frame_equal(frozen_rows, current_rows, check_dtype=False)
    print("Existing frozen v1 split verified. No files were overwritten.")
else:
    SPLIT_DIR.mkdir(parents=True, exist_ok=False)
    try:
        split_manifest.to_csv(private_csv, index=False)
        with private_json.open("x", encoding="utf-8") as out:
            json.dump(config, out, indent=2)
        with public_json.open("x", encoding="utf-8") as out:
            json.dump(summary_public, out, indent=2)
    except Exception:
        raise RuntimeError(
            "Split freeze could not finish. Inspect the private split folder "
            "before rerunning; nothing will be overwritten automatically."
        )
    print("Created frozen split v1.")

print("Private manifest:", private_csv)
print("Private config:", private_json)
print("Aggregate counts:", public_json)
print("The test partition is now reserved for a single final evaluation.")


## Stage 2 checkpoint

After executing the Stage 2 cells, confirm that:

- The frozen split contains **all 338 images** and **all three labels** in each partition.
- The two candidate near-duplicate pairs are assigned to the same partition.
- A repeat execution reports **existing frozen v1 split verified** (never a new random split).
- No feature scaling, augmentation, CLAHE tuning, or CNN training has been performed using the test set.
- The training pipeline in Stage 3 will use this frozen manifest and preserve identical splits for **all** architectures and preprocessing comparisons.
- The project will report *image-level validation/test results; patient independence not verified*.

Next: **Stage 3 — baseline and custom CNN training**. We can start with a 128×128 RGB reference and a normalized grayscale custom CNN, then compare 224×224 and CLAHE without touching the test set.